# Conv2d : repérer un motif avec le même filtre local

Calculer une première sortie, puis déplacer le noyau sans changer ses poids.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#convolution-a-la-main).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Notre classifieur associait deux mesures globales à deux scores. Une carte temps-fréquence offre une autre information : les motifs apparaissent à certains instants et dans certaines bandes. Une convolution essaie le même petit filtre à plusieurs endroits, plutôt que d’apprendre une règle séparée pour chaque position.

Avant de traiter un spectrogramme, choisissons une grille 3×3 numérotée de 1 à 9. Les lignes représentent des bandes et les colonnes des instants ; ces nombres ne sont pas des dB. Le noyau 2×2 vaut [[1,0],[0,−1]].

| Bande / temps | t0 | t1 | t2 |
| --- | --- | --- | --- |
| f0 | 1 | 2 | 3 |
| f1 | 4 | 5 | 6 |
| f2 | 7 | 8 | 9 |

La fenêtre supérieure gauche est [[1,2],[4,5]].

Les produits sont 1×1, 2×0, 4×0 et 5×(−1). Leur somme vaut −4. On déplace ensuite la fenêtre d’une colonne, puis d’une ligne, toujours avec les mêmes poids. Ici les quatre sommes valent −4 ; ce résultat vient des nombres choisis, pas d’une règle générale.

Les deux premiers 1 désignent les canaux d’entrée et de sortie. kernel_size=2 choisit le carré 2×2 ; stride=1 le déplace d’une case ; padding=0 n’ajoute pas de bord. Sans biais, nous suivons seulement les produits. Avec un biais, une valeur par filtre s’ajouterait à chaque position de sa sortie.

La première sortie devient −3 : seule la contribution de la case 1 a augmenté de 1. Les autres positions n’augmentent pas toutes autant, car elles lisent d’autres cases. La carte finale a la forme (1,1,2,2). PyTorch applique ici le noyau sans le retourner : c’est la convention de corrélation croisée de Conv2d.

Sur une onde, Conv1d applique la même idée sur le temps. Une entrée (4,1,100) et huit filtres de longueur 5, sans padding, donnent (4,8,96) : 100−5+1 positions. Les huit sorties sont des filtres, pas huit classes ni huit microphones. Cette règle de longueur s’étend aux deux directions d’une carte.

## Démonstration 1 · Un calcul local, une modification et le cas d’une onde

Retrouver les objets et exécuter les fragments du chapitre dans leur contexte complet.

### Préparer les bibliothèques

Les quatre axes rendent explicites l’unique extrait et l’unique canal.

In [3]:
import torch
from torch import nn


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [4]:
carte = torch.tensor([[[[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]]]])
noyau = torch.tensor([[[[1., 0.], [0., -1.]]]])


In [5]:
observer('carte', carte)
observer('noyau', noyau)

carte forme (1, 1, 3, 3) type torch.float32 requires_grad False
  valeurs : [[[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]]]
noyau forme (1, 1, 2, 2) type torch.float32 requires_grad False
  valeurs : [[[[1.0, 0.0], [0.0, -1.0]]]]


### Calculer l’état suivant

Le calcul local précède l’appel de couche.

In [6]:
fenetre = carte[0, 0, :2, :2]
produits = fenetre * noyau[0, 0]


In [7]:
observer('fenetre', fenetre)
observer('produits', produits)

fenetre forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 2.0], [4.0, 5.0]]
produits forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 0.0], [0.0, -5.0]]


### Calculer l’état suivant

Les poids sont imposés sans autograd, puis le même noyau parcourt quatre positions.

In [8]:
couche = nn.Conv2d(1, 1, kernel_size=2, stride=1, padding=0, bias=False)


In [9]:
observer('couche', couche)

### Appliquer le contexte explicite

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [10]:
with torch.no_grad():
    couche.weight.copy_(noyau)


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [11]:
sortie = couche(carte)


In [12]:
observer('sortie', sortie)

sortie forme (1, 1, 2, 2) type torch.float32 requires_grad True
  valeurs : [[[[-4.0, -4.0], [-4.0, -4.0]]]]


### Appliquer le contexte explicite

Le premier poids passe de 1 à 2 ; les autres restent fixés.

In [13]:
with torch.no_grad():
    couche.weight[0, 0, 0, 0] = 2.


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [14]:
variante = couche(carte)


In [15]:
observer('variante', variante)

variante forme (1, 1, 2, 2) type torch.float32 requires_grad True
  valeurs : [[[[-3.0, -2.0], [0.0, 1.0]]]]


### Calculer l’état suivant

Quatre signaux nuls isolent les tailles ; aucune mesure audio réelle n’est utilisée.

In [16]:
onde = torch.zeros(4, 1, 100)
conv_1d = nn.Conv1d(1, 8, kernel_size=5, bias=False)
sortie_1d = conv_1d(onde)


In [17]:
observer('onde', onde)
observer('conv_1d', conv_1d)
observer('sortie_1d', sortie_1d)

onde forme (4, 1, 100) type torch.float32 requires_grad False
sortie_1d forme (4, 8, 96) type torch.float32 requires_grad True


### Lire les sorties

Les sorties comparent le calcul manuel et les appels des couches.

In [18]:
print("produits locaux :", produits.tolist())
print("premiere somme :", produits.sum().item())
print("sortie :", sortie[0, 0].tolist(), "forme", tuple(sortie.shape))
print("poids modifie :", variante[0, 0].tolist())


produits locaux : [[1.0, 0.0], [0.0, -5.0]]
premiere somme : -4.0
sortie : [[-4.0, -4.0], [-4.0, -4.0]] forme (1, 1, 2, 2)
poids modifie : [[-3.0, -2.0], [0.0, 1.0]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [19]:
print("Conv1d :", tuple(sortie_1d.shape))


Conv1d : (4, 8, 96)


In [20]:
assert bool(produits.tolist() == [[1.,0.],[0.,-5.]] and produits.sum().item() == -4.), 'convolution-a-la-main/main: produits.tolist() == [[1.,0.],[0.,-5.]] and produits.sum().item() == -4.'
assert bool(tuple(sortie.shape) == (1,1,2,2) and sortie[0,0].tolist() == [[-4.,-4.],[-4.,-4.]]), 'convolution-a-la-main/main: tuple(sortie.shape) == (1,1,2,2) and sortie[0,0].tolist() == [[-4.,-4.],[-4.,-4.]]'
assert bool(variante[0,0].tolist() == [[-3.,-2.],[0.,1.]] and tuple(sortie_1d.shape) == (4,8,96)), 'convolution-a-la-main/main: variante[0,0].tolist() == [[-3.,-2.],[0.,1.]] and tuple(sortie_1d.shape) == (4,8,96)'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Calculez le premier résultat d’un noyau [[1,0],[0,−1]] appliqué à la grille 1 à 9.

In [21]:
exo_x = torch.arange(1.,10.).reshape(1,1,3,3)
exo_noyau = torch.tensor([[[[1.,0.],[0.,-1.]]]])
exo_valeur = ...  # Remplacer les points par votre réponse.
if exo_valeur is Ellipsis or exo_valeur is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_valeur)-torch.nn.functional.conv2d(exo_x,exo_noyau)[0,0,0,0].item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La première fenêtre est [[1,2],[4,5]]. Faites les quatre produits, puis leur somme.

</details>

### Exercice 2

Avec le même noyau 2×2 et stride=2, quelle forme de sortie reste-t-il sur une entrée 3×3 ?

In [22]:
exo_x = torch.arange(1.,10.).reshape(1,1,3,3)
exo_noyau = torch.ones(1,1,2,2)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.nn.functional.conv2d(exo_x,exo_noyau,stride=2).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le premier placement couvre indices 0 et 1. Un nouveau départ à 2 dépasserait le bord sans padding.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.arange(1.,10.).reshape(1,1,3,3)
exo_noyau = torch.tensor([[[[1.,0.],[0.,-1.]]]])
exo_valeur = -4.
if exo_valeur is Ellipsis or exo_valeur is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_valeur)-torch.nn.functional.conv2d(exo_x,exo_noyau)[0,0,0,0].item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.arange(1.,10.).reshape(1,1,3,3)
exo_noyau = torch.ones(1,1,2,2)
exo_forme = (1,1,1,1)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.nn.functional.conv2d(exo_x,exo_noyau,stride=2).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [25]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()